# Fig. 10: robustness to noisy CSI

Run cells in order from a fresh kernel. Settings are in `configs/experiments.json`; artifacts use the paths described in the README. Timings depend on hardware and software.


In [ ]:
from fthbf.runtime import (DATA_DIR, CHECKPOINT_DIR, checkpoint_path, load_artifact,
                           result_dir, read_config, seed_everything, record_run)
import torch
cfg = read_config("experiments.json")["Fig05"]
seed_everything(cfg["seed"])
record_run("Fig05", cfg)


In [ ]:
with torch.no_grad():
    # =====================================================
    # Robustness Test: WSR vs Target NMSE (precisely controlled)
    # =====================================================
    import os
    import copy
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    from solutions.trihybrid_WMMSE import TriHybridWMMSE, GNN
    from solutions.utils import computeWSR, channelDataset, HybridPrecodingBFGS, checkHybridConstraints

    # =====================================================
    # SETTINGS
    # =====================================================
    cdtype = torch.cfloat
    NUM_SAMPLES = cfg["samples"]
    Nrf = cfg["rf_chains"]
    UWMMSE_L1 = cfg["unfolded_layers"][0]
    UWMMSE_L2 = cfg["unfolded_layers"][1]
    WMMSE_L1 = cfg["classical_iterations"][0]
    WMMSE_L2 = cfg["classical_iterations"][1]
    num_points = cfg["nmse_points"]
    TARGET_NMSE_LIST = np.logspace(np.log10(1e-4), np.log10(0.5), num_points)

    RESULT_DIR = result_dir("results_WSR_vs_NMSE")
    os.makedirs(RESULT_DIR, exist_ok=True)
    SEED = cfg["seed"]
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # =====================================================
    # LOAD DATA
    # =====================================================
    SAVE_DIR = DATA_DIR
    parameters = load_artifact(os.path.join(SAVE_DIR, "parameters.pt"))
    Nt = parameters["Nt"]
    Nc = parameters["Nc"]
    P = parameters["P"]
    Mk = parameters["Mk"]
    eta = parameters["eta"]
    Ns_k = parameters["Ns_k"]
    sigma2 = parameters["sigma2"]

    data = load_artifact(os.path.join(SAVE_DIR, "probInsts_test.pt"))

    # =====================================================
    # NMSE precise corruption
    # =====================================================
    def add_channel_error_nmse(H_list, target_nmse):
        if target_nmse == 0:
            return [H.clone() for H in H_list]

        H_noisy = []

        for H in H_list:
            E = torch.randn_like(H) + 1j * torch.randn_like(H)

            norm_H2 = torch.linalg.norm(H)**2
            norm_E2 = torch.linalg.norm(E)**2

            alpha = torch.sqrt(target_nmse * norm_H2 / norm_E2)
            H_noisy.append(H + alpha * E)

        return H_noisy

    def compute_nmse(H_true, H_est):
        num = 0.0
        den = 0.0
        for Ht, He in zip(H_true, H_est):
            num += torch.linalg.norm(Ht - He)**2
            den += torch.linalg.norm(Ht)**2
        return (num / den).real.item()

    # =====================================================
    # UWMMSE MODEL LOADING
    # =====================================================
    def load_uwmmse_models(I_max):
        in_dim = 2 * data["ProbInsts"][0]["H"][0].numel() + 1
        GNNs_gamma = torch.nn.ModuleList()
        GNNs_u = torch.nn.ModuleList()

        for _ in range(I_max):
            GNNs_gamma.append(GNN(in_dim, 2*Ns_k**2+4, 2*Ns_k**2, 3, 0.1))
            GNNs_u.append(GNN(in_dim, Nt+10, Nt, 3, 0.1))

        ckpt = load_artifact(checkpoint_path(I_max), map_location="cpu")

        for i in range(I_max):
            GNNs_gamma[i].load_state_dict(ckpt["GNNs_gamma_state_dict"][i])
            GNNs_u[i].load_state_dict(ckpt["GNNs_u_state_dict"][i])

        for m in list(GNNs_gamma) + list(GNNs_u):
            m.eval()

        return GNNs_gamma, GNNs_u

    UWMMSE_MODEL1 = load_uwmmse_models(UWMMSE_L1)
    UWMMSE_MODEL2 = load_uwmmse_models(UWMMSE_L2)

    # =====================================================
    # COMMON HYBRID POST
    # =====================================================
    def hybrid_postprocess(FD_opt):
        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(max_iter=100, use_B0=True, verbose=False)
        _, papc_ok, info = checkHybridConstraints(FRF, FBB, P, tol=1e-6, verbose=False)
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))
        return FRF, FBB

    # =====================================================
    # ALGORITHMS
    # =====================================================
    def run_classical(prob_inst, H_input, H_true, I_max):
        K = prob_inst["K"]
        Ns = Ns_k * K
        u0 = [torch.ones(Nc,1,dtype=cdtype)]*Nt
        v0 = [torch.ones(Ns,1,dtype=cdtype)]*Nt
        for n in range(Nt):  # Match the power projection used during training.
            norm_vn = torch.linalg.norm(v0[n])
            if norm_vn > torch.sqrt(P):
                v0[n] = v0[n] / norm_vn * torch.sqrt(P)

        solver = TriHybridWMMSE(
            H_list=H_input,
            Ns_k=[Ns_k]*K,
            Mk=[Mk]*K,
            sigma2=[sigma2]*K,
            beta=[1.0]*K,
            P=[P]*Nt,
            eta=eta,
            u_init=u0,
            v_init=v0)

        _, FA, FD, _, _ = solver.run_classical(I_max=I_max, verbose=False)
        FRF, FBB = hybrid_postprocess(FD)
        R, _ = computeWSR(FA, FRF, FBB, H_true, parameters)
        return R.item()

    def run_UWMMSE(idx, dataset_noisy, H_input, H_true, I_max, unfolded_model):
        prob_inst = dataset_noisy[idx]

        K = prob_inst["K"]
        Ns = Ns_k*K
        u0 = [torch.ones(Nc,1,dtype=cdtype)]*Nt
        v0 = [torch.ones(Ns,1,dtype=cdtype)]*Nt
        for n in range(Nt):  # Match the power projection used during training.
            norm_vn = torch.linalg.norm(v0[n])
            if norm_vn > torch.sqrt(P):
                v0[n] = v0[n] / norm_vn * torch.sqrt(P)

        node_features = prob_inst["node_feature"]
        edge_index = prob_inst["edge_index"]
        graph_id = torch.zeros(K, dtype=torch.int32)
        num_graphs = torch.tensor(1, dtype=torch.int32)

        Gg, Gu = unfolded_model
        dgamma, du = [], []

        for i in range(I_max):
            dgamma.append(Gg[i](node_features, edge_index))
            du.append(Gu[i](node_features, edge_index, graph_id, num_graphs).reshape(Nt,1))

        solver = TriHybridWMMSE(
            H_list=H_input,
            Ns_k=[Ns_k]*K,
            Mk=[Mk]*K,
            sigma2=[sigma2]*K,
            beta=[1.0]*K,
            P=[P]*Nt,
            eta=eta,
            u_init=u0,
            v_init=v0)

        _, FA, FD, _, _ = solver.run_unfolded(dgamma, du, I_max=I_max, verbose=False)
        FRF, FBB = hybrid_postprocess(FD)
        R, _ = computeWSR(FA, FRF, FBB, H_true, parameters)
        return R.item()

    # =====================================================
    # BENCHMARK
    # =====================================================
    indices = np.random.choice(len(data["ProbInsts"]), NUM_SAMPLES, replace=False)
    ALGS = ([f"Unfolded Iter={UWMMSE_L1}"] + [f"Unfolded Iter={UWMMSE_L2}"] + [f"Model-Based Iter={WMMSE_L1}"] + [f"Model-Based Iter={WMMSE_L2}"])
    results = {a: [] for a in ALGS}
    actual_nmse = []
    for target_nmse in TARGET_NMSE_LIST:

        # Add noise to the complete dataset
        data_noisy = copy.deepcopy(data)
        for ind_prob in range(len(data["ProbInsts"])):
            H_true_list = data["ProbInsts"][ind_prob]["H"]
            H_noisy_list = add_channel_error_nmse(H_true_list, target_nmse)
            data_noisy["ProbInsts"][ind_prob]["H"] = H_noisy_list
        ProbInsts_noisy = data_noisy["ProbInsts"]
        dataset_noisy = channelDataset(ProbInsts_noisy, parameters)

        # Run algorithms on the noisy dataset
        store = {alg: [] for alg in ALGS}
        nmse_samples = []
        for idx in tqdm(indices):
            prob_inst = dataset_noisy[idx]
            H_list_true = data["ProbInsts"][idx]["H"]
            H_list_noisy = prob_inst["H_list"]
            nmse_sample = compute_nmse(H_list_true, H_list_noisy)
            nmse_samples.append(nmse_sample)
            # Model-based, Iter = WMMSE_L1
            store[f"Model-Based Iter={WMMSE_L1}"].append(run_classical(prob_inst, H_list_noisy, H_list_true, WMMSE_L1))
            # Model-based, Iter = WMMSE_L2
            store[f"Model-Based Iter={WMMSE_L2}"].append(run_classical(prob_inst, H_list_noisy, H_list_true, WMMSE_L2))
            # Unfolded, Iter = UWMMSE_L1
            store[f"Unfolded Iter={UWMMSE_L1}"].append(run_UWMMSE(idx, dataset_noisy, H_list_noisy, H_list_true, UWMMSE_L1, UWMMSE_MODEL1))
            # Unfolded, Iter = UWMMSE_L2
            store[f"Unfolded Iter={UWMMSE_L2}"].append(run_UWMMSE(idx, dataset_noisy, H_list_noisy, H_list_true, UWMMSE_L2, UWMMSE_MODEL2))
        actual_nmse.append(np.mean(nmse_samples))
        print(
            f"Target NMSE={target_nmse:.2e} | "
            f"Actual mean={np.mean(nmse_samples):.2e} | "
            f"std={np.std(nmse_samples):.2e}")
        for alg in ALGS:
            results[alg].append(np.array(store[alg]))


    # Save results
    save_path = os.path.join(RESULT_DIR, "wsr_vs_nmse_results.npz")
    np.savez(
        save_path,
        target_nmse=TARGET_NMSE_LIST,
        actual_nmse=np.array(actual_nmse),
        **{alg: np.stack(results[alg]) for alg in ALGS})
    print(f"\nResults saved to {save_path}")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

RESULT_DIR = result_dir("results_WSR_vs_NMSE")
file_path = os.path.join(RESULT_DIR, "wsr_vs_nmse_results.npz")

data = np.load(file_path)

target_nmse = data["target_nmse"]
actual_nmse = data["actual_nmse"]

alg_names = ([f"Unfolded Iter={UWMMSE_L1}"] + [f"Unfolded Iter={UWMMSE_L2}"])

plt.figure(figsize=(7,5))

for alg in alg_names:
    wsr = data[alg]              # shape (L, samples)
    mean_wsr = wsr.mean(axis=1)
    std_wsr = wsr.std(axis=1)

    plt.plot(actual_nmse, mean_wsr, marker='o', label=alg)
    plt.fill_between(actual_nmse, mean_wsr - std_wsr, mean_wsr + std_wsr, alpha=0.2)
    
alg_model = f"Model-Based Iter={WMMSE_L1}"
plt.plot(actual_nmse, data[alg_model].mean(axis=1), marker='s', linestyle='--', color='black', label=alg_model)
alg_model = f"Model-Based Iter={WMMSE_L2}"
plt.plot(actual_nmse, data[alg_model].mean(axis=1), marker='^', linestyle='--', color='black', label=alg_model)

plt.xscale("log")
plt.xlabel("NMSE")
plt.ylabel("Average WSR (bit/s/Hz)")
plt.title("Robustness: WSR vs Channel Estimation Error")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.tight_layout()

save_fig = os.path.join(RESULT_DIR, "WSR_vs_NMSE.png")
plt.savefig(save_fig, dpi=300)
plt.show()

print(f"Figure saved to {save_fig}")

In [ ]:
import os
import numpy as np

RESULT_DIR = result_dir("results_WSR_vs_NMSE")
file_path = os.path.join(RESULT_DIR, "wsr_vs_nmse_results.npz")

data = np.load(file_path)

actual_nmse = data["actual_nmse"]


# =====================================================
# TikZ helpers
# =====================================================
def print_table(x, y):
    for xi, yi in zip(x, y):
        print(f"{xi:.6e}\t{yi:.6f}\\\\")


def print_mean_std_curve(x, mean, std, legend,
                         color="tabblue",
                         mark="o",
                         name="curve"):
    """
    Outputs:
        upper curve
        lower curve
        fill_between
        mean curve
    """

    upper = mean + std
    lower = mean - std

    # ---------- upper ----------
    print(f"% ---- {legend} upper ----")
    print(f"\\addplot[name path={name}_upper, draw=none] table[row sep=crcr]{{%")
    print_table(x, upper)
    print("};")

    # ---------- lower ----------
    print(f"% ---- {legend} lower ----")
    print(f"\\addplot[name path={name}_lower, draw=none] table[row sep=crcr]{{%")
    print_table(x, lower)
    print("};")

    # ---------- fill ----------
    print(f"% ---- {legend} band ----")
    print(
        f"\\addplot[{color}!20, opacity=0.4] "
        f"fill between[of={name}_upper and {name}_lower];"
    )

    # ---------- mean line ----------
    print(f"% ---- {legend} mean ----")
    print(
        f"\\addplot["
        f"color={color}, "
        f"mark={mark}, "
        f"mark size=2pt, "
        f"line width=1.2pt"
        f"] table[row sep=crcr]{{%"
    )
    print_table(x, mean)
    print("};")
    print(f"\\addlegendentry{{{legend}}}")
    print()


def print_mean_curve(x, mean, legend,
                     color="black",
                     mark="square*"):
    print(f"% ---- {legend} ----")
    print(
        f"\\addplot["
        f"color={color}, "
        f"mark={mark}, "
        f"mark size=2pt, "
        f"line width=1.2pt"
        f"] table[row sep=crcr]{{%"
    )
    print_table(x, mean)
    print("};")
    print(f"\\addlegendentry{{{legend}}}")
    print()


# =====================================================
# Print TikZ body
# =====================================================

print("\n% =======================================")
print("% TikZ curves (paste inside axis)")
print("% =======================================\n")

# -------- unfolded curves --------
for key in data.files:

    if key.startswith("Unfolded"):

        wsr = data[key]
        mean = wsr.mean(axis=1)
        std = wsr.std(axis=1)

        # Plot appearance
        color = "tabblue"
        mark = "o"

        if "2" in key:
            color = "tabblue"
            mark = "*"
        elif "4" in key:
            color = "taborange"
            mark = "triangle*"
        elif "6" in key:
            color = "tabgreen"
            mark = "diamond*"

        print_mean_std_curve(
            actual_nmse,
            mean,
            std,
            legend=key,
            color=color,
            mark=mark,
            name=key.replace(" ", "_").replace("=", "")
        )


# -------- model-based --------
for key in data.files:
    if key.startswith("Model-Based"):
        mean = data[key].mean(axis=1)
        print_mean_curve(actual_nmse, mean, key, color="black", mark="square*")